# MAIRA-2 — Génération de rapport ancré, localisation et classification dérivée

**Projet ARV-DS03 — approfondissement : modèle médical spécialisé MAIRA-2 (Microsoft)**

MAIRA-2 est un modèle multimodal (encodeur RAD-DINO gelé + Vicuna-7B fine-tuné) qui **génère des comptes-rendus radiologiques ancrés** : chaque observation est accompagnée de boîtes englobantes localisant l'anomalie. Contrairement à Gemma 4, il s'utilise **par prompting** (aucun fine-tuning), dans son format d'entrée prévu.

Ce notebook :
1. génère un rapport ancré par image (localisation native, vraies boîtes) ;
2. **dérive une classe** (normal / suspected_opacity / uncertain) par post-traitement du texte du rapport ;
3. évalue la **classification** (vs vérité RSNA) et la **localisation** (accord par quadrant, comparable à la partie B Gemma).

> Accès : MAIRA-2 est un modèle *gated*. Il faut avoir demandé l'accès sur sa page Hugging Face et enregistré un secret Kaggle `HF_TOKEN` (Add-ons → Secrets).

## 1. Environnement

MAIRA-2 exige une version précise de `transformers` (testé en 4.51.3) et `trust_remote_code`. On épingle la version pour éviter les incompatibilités du code distant.

In [1]:
!pip install -q "transformers==4.51.3" "bitsandbytes>=0.43.0" pillow protobuf sentencepiece 2>/dev/null

# Diagnostic : verifier que bitsandbytes s'importe VRAIMENT (pas juste installe)
try:
    import bitsandbytes as bnb
    print("bitsandbytes importe :", bnb.__version__)
except Exception as e:
    print("ECHEC import bitsandbytes :", e)

from transformers.utils import is_bitsandbytes_available
print("transformers voit bitsandbytes :", is_bitsandbytes_available())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 78.0 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 85.8 MB/s eta 0:00:00:00:01
bitsandbytes importe : 0.49.2
transformers voit bitsandbytes : True


In [2]:
import os, json, re, torch, numpy as np, pandas as pd, cv2
from PIL import Image
import pydicom
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# Authentification Hugging Face (secret Kaggle "HF_TOKEN" requis : modele gated)
login(UserSecretsClient().get_secret("HF_TOKEN"))

import transformers
print("transformers :", transformers.__version__)
print("CUDA         :", torch.cuda.get_device_name(0))

transformers : 4.51.3
CUDA         : Tesla T4


## 2. Dataset RSNA (labels + boîtes)

Mêmes données que les notebooks Gemma, pour que les comparaisons soient valides. On garde le mapping trois classes pour l'évaluation de classification, et les boîtes `Target=1` pour la localisation.

In [3]:
import glob
matches = glob.glob('/kaggle/input/**/stage_2_detailed_class_info.csv', recursive=True)
if not matches:
    raise FileNotFoundError("Dataset RSNA introuvable (Add Input).")
BASE_DIR = os.path.dirname(matches[0]) + '/'
TRAIN_IMAGES_DIR = os.path.join(BASE_DIR, 'stage_2_train_images')

df_class = pd.read_csv(os.path.join(BASE_DIR, 'stage_2_detailed_class_info.csv')).drop_duplicates('patientId').reset_index(drop=True)
CLASS_MAP = {
    "Normal": "normal",
    "Lung Opacity": "suspected_opacity",
    "No Lung Opacity / Not Normal": "uncertain",
}
df_class["label"] = df_class["class"].map(CLASS_MAP)
df_boxes = pd.read_csv(os.path.join(BASE_DIR, 'stage_2_train_labels.csv'))

from sklearn.model_selection import train_test_split
# Pas d'entrainement : un seul echantillon d'evaluation suffit (prompting pur).
_, eval_sub = train_test_split(df_class, test_size=0.20, stratify=df_class["label"], random_state=42)
N_EVAL = 100  # ajustable ; MAIRA-2 est lourd, on commence petit
eval_sub, _ = train_test_split(eval_sub, train_size=N_EVAL, stratify=eval_sub["label"], random_state=42)
eval_sub = eval_sub.reset_index(drop=True)
print("Echantillon eval :", len(eval_sub))
print(eval_sub["label"].value_counts())

Echantillon eval : 100
label
uncertain            44
normal               33
suspected_opacity    23
Name: count, dtype: int64


## 3. Chargement des images

MAIRA-2 attend une image PIL RGB. On lit le DICOM, on normalise et on applique CLAHE (rehaussement de contraste), comme pour Gemma. On ne redimensionne pas : le processeur de MAIRA-2 s'en charge (et fournit de quoi réajuster les boîtes).

In [4]:
def load_cxr(patient_id):
    """DICOM -> image PIL RGB (sans resize : le processeur MAIRA-2 recadre lui-meme)."""
    path = os.path.join(TRAIN_IMAGES_DIR, f"{patient_id}.dcm")
    if not os.path.exists(path):
        return None
    dcm = pydicom.dcmread(path)
    img = dcm.pixel_array.astype(np.float32)
    img = (img - img.min()) / (img.max() - img.min() + 1e-8) * 255.0
    img = img.astype(np.uint8)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    img = clahe.apply(img)
    return Image.fromarray(cv2.cvtColor(img, cv2.COLOR_GRAY2RGB))

_t = load_cxr(eval_sub.iloc[0]["patientId"])
print("Image test :", _t.size, _t.mode)

Image test : (1024, 1024) RGB


## 4. Chargement de MAIRA-2 (4-bit pour tenir sur T4)

Le modèle fait ~7 Md de paramètres. En précision native il ne tient pas sur un T4 (16 Go) avec la marge nécessaire à l'inférence. On le charge en **4-bit** (quantisation), comme pour Gemma.

In [5]:
from transformers import AutoModelForCausalLM, AutoProcessor, BitsAndBytesConfig

MODEL_ID = "microsoft/maira-2"
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    quantization_config=bnb,
    device_map={"": 0},
)
model = model.eval()
print("MAIRA-2 charge en 4-bit.")

processor_config.json:   0%|          | 0.00/405 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/611 [00:00<?, ?B/s]

processing_maira2.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/maira-2:
- processing_maira2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


preprocessor_config.json:   0%|          | 0.00/560 [00:00<?, ?B/s]

...

model-00001-of-00006.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

model-00003-of-00006.safetensors:   0%|          | 0.00/4.86G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/179 [00:00<?, ?B/s]

MAIRA-2 charge en 4-bit.


## 5. Génération d'un rapport ancré

On utilise l'API prévue par MAIRA-2 : `format_and_preprocess_reporting_input` avec `get_grounding=True`. On ne fournit que la vue frontale (RSNA n'a pas de vue latérale ni d'antériorité). On ne modifie **pas** le format d'instruction : le modèle y est très sensible.

La sortie est une liste de couples `(phrase, [boîtes])`, les boîtes étant en coordonnées relatives à l'image **recadrée** vue par le modèle.

In [6]:
def generate_grounded_report(image):
    """Retourne la sortie ancree parsee : liste de (phrase, boites|None)."""
    processed = processor.format_and_preprocess_reporting_input(
        current_frontal=image,
        current_lateral=None,
        prior_frontal=None,
        indication=None,
        technique=None,
        comparison=None,
        prior_report=None,
        return_tensors="pt",
        get_grounding=True,   # rapport ANCRE (avec boites)
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(**processed, max_new_tokens=450, use_cache=True)
    prompt_len = processed["input_ids"].shape[-1]
    text = processor.decode(out[0][prompt_len:], skip_special_tokens=True).lstrip()
    # Parsing officiel MAIRA-2 -> sequence structuree (phrase, boites)
    parsed = processor.convert_output_to_plaintext_or_grounded_sequence(text)
    return parsed, text

# Test rapide sur une image
demo_parsed, demo_raw = generate_grounded_report(load_cxr(eval_sub.iloc[0]["patientId"]))
print("Sortie brute :\n", demo_raw[:500], "\n")
print("Sortie parsee :\n", demo_parsed)

Sortie brute :
 <obj> Left picc line, tip in the svc.<box><x38><y18><x76><y38></box></obj><obj> The heart size is normal.</obj><obj> The lung fields are clear.</obj><obj> No focal infiltrates.</obj><obj> No effusions.</obj><obj> No significant bony abnormalities.</obj> 

Sortie parsee :
 [('Left picc line, tip in the svc.', [(0.385, 0.185, 0.765, 0.385)]), ('The heart size is normal.', None), ('The lung fields are clear.', None), ('No focal infiltrates.', None), ('No effusions.', None), ('No significant bony abnormalities.', None)]


## 6. Dérivation de la classe à partir du rapport (post-traitement)

MAIRA-2 ne produit pas de classe : il rédige un compte-rendu. On **dérive** la classe des trois catégories du projet à partir du texte, par détection de mots-clés radiologiques.

Logique (prudente, alignée sur la priorité dépistage) :
- présence de termes d'**opacité / consolidation / pneumonie / infiltrat** → `suspected_opacity` ;
- rapport explicitement **normal / clair / sans anomalie** → `normal` ;
- sinon (rapport non concluant ou ambigu) → `uncertain` (abstention).

C'est une heuristique documentée, à assumer comme telle : elle traduit un texte libre en étiquette, et constitue une source d'erreur distincte du modèle lui-même.

In [ ]:
import re

# --- Termes indiquant une ANOMALIE de type opacite (findings pulmonaires) ---
OPACITY_TERMS = [
    "opacity", "opacities", "opacification",
    "consolidation", "consolidations",
    "pneumonia", "pneumonic",
    "infiltrate", "infiltrates", "infiltration",
    "airspace disease", "air-space disease", "air space disease",
    "effusion", "effusions",
    "atelectasis", "atelectatic",
    "edema", "pulmonary edema",
    "haziness", "hazy",
    "density", "densities",
    "mass", "nodule", "nodules",
]

# --- Termes de NORMALITE explicite ---
NORMAL_TERMS = [
    "clear", "unremarkable", "no acute", "no focal", "normal",
    "no evidence", "within normal limits", "no abnormal",
]

# --- Marqueurs de NEGATION (si presents juste avant un terme, on l'annule) ---
NEGATION_CUES = [
    "no", "not", "without", "negative for", "free of", "clear of",
    "resolution of", "resolved", "absence of", "rule out", "ruled out",
]

# --- Phrases decrivant des DISPOSITIFS / structures NON pathologiques ---
# (une boite sur ces phrases ne doit PAS compter comme opacite)
DEVICE_TERMS = [
    "picc", "line", "tube", "catheter", "pacemaker", "lead", "wire",
    "sternotomy", "clip", "device", "port", "valve", "stent",
    "et tube", "ng tube", "tracheostomy",
]

def _negated(text, term):
    """Vrai si `term` apparait dans `text` precede (a <=4 mots) d'une negation."""
    for m in re.finditer(re.escape(term), text):
        start = m.start()
        # fenetre de contexte : ~40 caracteres avant le terme
        window = text[max(0, start-40):start]
        if any(cue in window for cue in NEGATION_CUES):
            return True
    return False

def _term_present_affirmative(text, terms):
    """Vrai s'il existe au moins un terme de la liste present ET non nie."""
    for t in terms:
        if t in text and not _negated(text, t):
            return True
    return False

def phrase_is_pathological(phrase):
    """Une phrase decrit-elle une anomalie pulmonaire (vs dispositif/normalite) ?"""
    p = phrase.lower()
    if any(d in p for d in DEVICE_TERMS):
        return False  # dispositif medical : pas une opacite
    if _term_present_affirmative(p, OPACITY_TERMS):
        return True
    return False

def derive_class(parsed_output):
    """
    Classe derivee en DEUX niveaux :
      1) signal fort : une BOITE ancree sur une phrase pathologique => suspected_opacity
      2) signal texte : termes d'opacite non nies => suspected_opacity
                        sinon normalite explicite => normal
                        sinon => uncertain
    Retourne (classe, rapport_texte, a_une_boite_pathologique)
    """
    if not isinstance(parsed_output, (list, tuple)) or len(parsed_output) == 0:
        return "uncertain", "", False

    phrases = []
    box_on_pathology = False
    for item in parsed_output:
        if isinstance(item, (list, tuple)) and len(item) >= 1:
            phrase = str(item[0])
            boxes = item[1] if len(item) >= 2 else None
            phrases.append(phrase)
            # NIVEAU 1 : une boite posee sur une phrase pathologique
            if boxes and phrase_is_pathological(phrase):
                box_on_pathology = True
        else:
            phrases.append(str(item))

    report = " ".join(phrases).lower()

    # Niveau 1 : signal fort par la localisation
    if box_on_pathology:
        return "suspected_opacity", report, True

    # Niveau 2 : analyse texte avec gestion des negations
    if _term_present_affirmative(report, OPACITY_TERMS):
        return "suspected_opacity", report, False
    if _term_present_affirmative(report, NORMAL_TERMS):
        return "normal", report, False
    return "uncertain", report, False

# Test sur la demo
cls_demo, rep_demo, boxpath_demo = derive_class(demo_parsed)
print("Classe derivee   :", cls_demo)
print("Boite patho ?    :", boxpath_demo)
print("Rapport (extrait):", rep_demo[:300])


## 7. Extraction des boîtes pour la localisation

Pour comparer à la partie B de Gemma, on convertit les boîtes de MAIRA-2 en quadrant. Deux précautions :
- les boîtes sont **relatives à l'image recadrée** vue par le modèle ; le processeur fournit `adjust_box_for_original_image_size` pour repasser aux dimensions d'origine ;
- MAIRA-2 renvoie des coordonnées `(x_tl, y_tl, x_br, y_br)` normalisées (0–1).

In [8]:
def box_center_to_quadrant(x_tl, y_tl, x_br, y_br):
    """Boite normalisee (0-1) -> quadrant qualitatif (comme la partie B Gemma)."""
    cx = (x_tl + x_br) / 2
    cy = (y_tl + y_br) / 2
    horiz = "left" if cx < 0.5 else "right"
    if cy < 1/3:
        vert = "upper"
    elif cy < 2/3:
        vert = "middle"
    else:
        vert = "lower"
    return f"{vert}-{horiz}"

def first_box_quadrant(parsed_output):
    """Retourne le quadrant de la premiere boite trouvee dans la sortie ancree, ou None."""
    if not isinstance(parsed_output, (list, tuple)):
        return None
    for item in parsed_output:
        if isinstance(item, (list, tuple)) and len(item) >= 2 and item[1]:
            boxes = item[1]
            if boxes:
                b = boxes[0]
                if len(b) == 4:
                    return box_center_to_quadrant(*b)
    return None

# Verite terrain quadrant (comme partie B Gemma)
def gt_quadrant(patient_id):
    rows = df_boxes[(df_boxes["patientId"] == patient_id) & (df_boxes["Target"] == 1)]
    if len(rows) == 0:
        return None
    r = rows.iloc[0]
    dcm = pydicom.dcmread(os.path.join(TRAIN_IMAGES_DIR, f"{patient_id}.dcm"))
    H, W = dcm.pixel_array.shape
    cx, cy = r["x"] + r["width"]/2, r["y"] + r["height"]/2
    horiz = "left" if cx < W/2 else "right"
    if cy < H/3: vert = "upper"
    elif cy < 2*H/3: vert = "middle"
    else: vert = "lower"
    return f"{vert}-{horiz}"

## 8. Évaluation complète

Boucle unique : pour chaque image d'éval, on génère le rapport ancré, on en dérive la classe et le quadrant. On calcule ensuite la classification (3 classes) et l'accord de localisation (sur les vraies opacités uniquement).

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from tqdm.auto import tqdm

preds, trues, reports = [], [], []
loc_pred_q, loc_true_q = [], []
viz_records = []  # pour la visualisation (section suivante)

for _, row in tqdm(eval_sub.iterrows(), total=len(eval_sub)):
    pid = row["patientId"]
    img = load_cxr(pid)
    if img is None:
        preds.append("uncertain"); trues.append(row["label"]); reports.append("")
        continue
    try:
        parsed, raw = generate_grounded_report(img)
    except Exception:
        parsed, raw = [], ""
    cls, report, box_on_patho = derive_class(parsed)
    preds.append(cls); trues.append(row["label"]); reports.append(report)

    # Boites predites (toutes) pour la visualisation
    pred_boxes = []
    for item in parsed if isinstance(parsed, (list, tuple)) else []:
        if isinstance(item, (list, tuple)) and len(item) >= 2 and item[1]:
            for b in item[1]:
                if len(b) == 4:
                    pred_boxes.append((item[0], b))  # (phrase, boite normalisee)

    # Boites reelles (verite terrain RSNA)
    gt_rows = df_boxes[(df_boxes["patientId"] == pid) & (df_boxes["Target"] == 1)]

    viz_records.append({
        "pid": pid, "true": row["label"], "pred": cls,
        "report": report, "pred_boxes": pred_boxes,
        "gt_rows": gt_rows, "box_on_patho": box_on_patho,
        "raw": raw, "n_pred_boxes": len(pred_boxes),
    })

    # Localisation (sur vraies opacites)
    if row["label"] == "suspected_opacity":
        gtq = gt_quadrant(pid)
        pq = first_box_quadrant(parsed)
        if gtq is not None:
            loc_true_q.append(gtq)
            loc_pred_q.append(pq if pq is not None else "none")

labels = ["normal", "suspected_opacity", "uncertain"]
acc = accuracy_score(trues, preds)
f1  = f1_score(trues, preds, labels=labels, average="macro")
print("=== CLASSIFICATION (derivee du rapport MAIRA-2, v2 avec negations + boites) ===")
print(f"Accuracy : {acc:.4f}   Macro-F1 : {f1:.4f}")
print(classification_report(trues, preds, labels=labels, zero_division=0))
print(pd.DataFrame(confusion_matrix(trues, preds, labels=labels), index=labels, columns=labels))

cm = confusion_matrix(trues, preds, labels=labels)
op_i = labels.index("suspected_opacity")
tp = cm[op_i, op_i]; support = cm[op_i].sum()
if support:
    print(f"\n>>> Rappel suspected_opacity : {tp/support:.4f}")

if loc_true_q:
    exact = np.mean([p == t for p, t in zip(loc_pred_q, loc_true_q)])
    side  = np.mean([p.split("-")[-1] == t.split("-")[-1]
                     for p, t in zip(loc_pred_q, loc_true_q) if p != "none"])
    detected = np.mean([p != "none" for p in loc_pred_q])
    print("\n=== LOCALISATION ===")
    print(f"Cas evalues      : {len(loc_true_q)}")
    print(f"Taux detection   : {detected:.4f}")
    print(f"Accord quadrant  : {exact:.4f}")
    print(f"Accord cote (H)  : {side:.4f}")

pd.DataFrame({"patientId": eval_sub["patientId"], "true": trues, "pred": preds,
              "report": reports}).to_csv("/kaggle/working/maira2_results.csv", index=False)
print("\nResultats sauvegardes.")


## 9. Notes pour le rapport

- **MAIRA-2 vs Gemma / MedGemma.** MAIRA-2 est un modèle *spécialisé rapport radiologique*, utilisé par prompting (aucun fine-tuning). La classe est **dérivée** d'un texte libre, ce qui ajoute une source d'erreur (l'heuristique de mots-clés) distincte du modèle.
- **Localisation native.** Contrairement à l'approche par quadrants demandée à Gemma, MAIRA-2 produit de **vraies boîtes**. On les a ramenées au quadrant uniquement pour comparer à la partie B — mais on pourrait aussi rapporter un IoU avec les boîtes RSNA, métrique plus fine (piste d'extension).
- **Sensibilité au format.** On a utilisé strictement l'API `format_and_preprocess_reporting_input` sans modifier l'instruction : MAIRA-2 se dégrade si on change son format d'entrée.
- **Limite honnête.** MAIRA-2 a été entraîné sur des rapports US/Espagne ; la distribution RSNA peut différer. Les résultats sont à interpréter comme un transfert hors distribution.

## 10. Visualisation : boîtes prédites (MAIRA-2) vs réelles (RSNA)

On affiche une dizaine d'images avec, superposées :
- en **vert** : les boîtes de vérité terrain RSNA (opacités annotées) ;
- en **rouge** : les boîtes prédites par MAIRA-2 (avec la phrase associée).

On privilégie les cas de vraies opacités (là où la comparaison est parlante).

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# On priorise les vraies opacites (boites GT presentes), puis on complete.
viz_sorted = sorted(viz_records, key=lambda r: 0 if r["true"] == "suspected_opacity" else 1)
to_show = viz_sorted[:10]

n = len(to_show)
cols = 2
rows_n = (n + cols - 1) // cols
fig, axes = plt.subplots(rows_n, cols, figsize=(13, 5.5 * rows_n))
axes = np.array(axes).reshape(-1)

for ax, rec in zip(axes, to_show):
    pid = rec["pid"]
    img = load_cxr(pid)
    W, H = img.size  # PIL : (width, height)
    ax.imshow(img, cmap="gray")

    # Boites reelles (RSNA) en vert — coordonnees en pixels de l'image d'origine
    dcm = pydicom.dcmread(os.path.join(TRAIN_IMAGES_DIR, f"{pid}.dcm"))
    oh, ow = dcm.pixel_array.shape
    sx, sy = W / ow, H / oh  # facteur d'echelle origine -> image affichee
    for _, gr in rec["gt_rows"].iterrows():
        rect = mpatches.Rectangle((gr["x"]*sx, gr["y"]*sy), gr["width"]*sx, gr["height"]*sy,
                                  linewidth=2.2, edgecolor="lime", facecolor="none")
        ax.add_patch(rect)

    # Boites predites (MAIRA-2) en rouge — coordonnees normalisees 0-1
    for phrase, b in rec["pred_boxes"]:
        x_tl, y_tl, x_br, y_br = b
        rect = mpatches.Rectangle((x_tl*W, y_tl*H), (x_br-x_tl)*W, (y_br-y_tl)*H,
                                  linewidth=2.0, edgecolor="red", facecolor="none", linestyle="--")
        ax.add_patch(rect)

    ok = "OK" if rec["true"] == rec["pred"] else "X"
    ax.set_title(f"[{ok}] vrai={rec['true']} | pred={rec['pred']}"
                 + (" | boite patho" if rec["box_on_patho"] else ""),
                 fontsize=10)
    ax.axis("off")

# Legende + masquer axes inutilises
for ax in axes[n:]:
    ax.axis("off")
green_patch = mpatches.Patch(color="lime", label="Verite RSNA (opacite reelle)")
red_patch   = mpatches.Patch(color="red", label="Prediction MAIRA-2")
fig.legend(handles=[green_patch, red_patch], loc="upper center", ncol=2, fontsize=11)
plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.savefig("/kaggle/working/maira2_visualisation.png", dpi=110, bbox_inches="tight")
plt.show()
print("Figure sauvegardee : /kaggle/working/maira2_visualisation.png")

# Afficher aussi les rapports texte des images montrees (utile pour l'analyse)
print("\n--- Rapports des images affichees ---")
for rec in to_show:
    print(f"\n[{rec['pid']}] vrai={rec['true']} pred={rec['pred']}")
    print("  ", rec["report"][:280])


## 11. Export Excel pour analyse externe (par une autre IA)

On exporte, pour les 100 images de test, l'intégralité de ce que MAIRA-2 a produit : rapport brut, rapport nettoyé, chaque phrase avec sa/ses boîte(s), la classe dérivée, la vérité terrain, et les boîtes réelles RSNA. Le fichier comporte deux feuilles :

- **Synthèse** : une ligne par image (vue d'ensemble, idéale pour un premier tri) ;
- **Détail_boîtes** : une ligne par boîte prédite (phrase + coordonnées), pour l'analyse fine de la localisation.

Ce format est pensé pour être fourni à un autre modèle qui analysera les diagnostics.

In [ ]:
import pandas as pd

# ---------- Feuille 1 : synthese (1 ligne / image) ----------
synth_rows = []
for rec in viz_records:
    pid = rec["pid"]
    # vérité terrain : boîtes réelles en texte
    gt_boxes = []
    for _, gr in rec["gt_rows"].iterrows():
        gt_boxes.append(f"(x={int(gr['x'])},y={int(gr['y'])},w={int(gr['width'])},h={int(gr['height'])})")
    gt_boxes_str = " ; ".join(gt_boxes) if gt_boxes else ""

    # boîtes prédites en texte (phrase -> coords normalisées)
    pred_boxes_str = " ; ".join(
        f"[{phrase}] ({x1:.2f},{y1:.2f},{x2:.2f},{y2:.2f})"
        for phrase, (x1, y1, x2, y2) in rec["pred_boxes"]
    )

    synth_rows.append({
        "patientId": pid,
        "verite_terrain": rec["true"],
        "classe_predite": rec["pred"],
        "correct": rec["true"] == rec["pred"],
        "boite_sur_pathologie": rec["box_on_patho"],
        "nb_boites_predites": rec["n_pred_boxes"],
        "nb_boites_reelles": len(rec["gt_rows"]),
        "rapport_maira": rec["report"],
        "rapport_brut": rec["raw"],
        "boites_predites": pred_boxes_str,
        "boites_reelles_RSNA": gt_boxes_str,
    })
df_synth = pd.DataFrame(synth_rows)

# ---------- Feuille 2 : detail des boites (1 ligne / boite predite) ----------
box_rows = []
for rec in viz_records:
    if not rec["pred_boxes"]:
        box_rows.append({
            "patientId": rec["pid"], "verite_terrain": rec["true"],
            "classe_predite": rec["pred"], "phrase": "(aucune boite)",
            "x_tl": None, "y_tl": None, "x_br": None, "y_br": None,
            "quadrant": None,
        })
        continue
    for phrase, (x1, y1, x2, y2) in rec["pred_boxes"]:
        cx, cy = (x1+x2)/2, (y1+y2)/2
        horiz = "left" if cx < 0.5 else "right"
        vert = "upper" if cy < 1/3 else ("middle" if cy < 2/3 else "lower")
        box_rows.append({
            "patientId": rec["pid"], "verite_terrain": rec["true"],
            "classe_predite": rec["pred"], "phrase": phrase,
            "x_tl": round(x1,3), "y_tl": round(y1,3),
            "x_br": round(x2,3), "y_br": round(y2,3),
            "quadrant": f"{vert}-{horiz}",
        })
df_boxes_detail = pd.DataFrame(box_rows)

# ---------- Feuille 3 : analyse d'erreurs (cas mal classes) ----------
df_err = df_synth[df_synth["correct"] == False][
    ["patientId", "verite_terrain", "classe_predite", "boite_sur_pathologie",
     "nb_boites_predites", "rapport_maira", "boites_predites"]
].copy()
# Focus particulier : les uncertain versés à tort dans suspected_opacity
df_err["type_erreur"] = df_err["verite_terrain"] + " -> " + df_err["classe_predite"]
df_err = df_err.sort_values("type_erreur").reset_index(drop=True)

# ---------- Ecriture du fichier Excel ----------
out_path = "/kaggle/working/maira2_analyse_complete.xlsx"
with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
    df_synth.to_excel(writer, sheet_name="Synthese", index=False)
    df_boxes_detail.to_excel(writer, sheet_name="Detail_boites", index=False)
    df_err.to_excel(writer, sheet_name="Analyse_erreurs", index=False)

    # Mise en forme legere : largeurs de colonnes + en-tetes en gras
    from openpyxl.styles import Font, PatternFill, Alignment
    header_fill = PatternFill("solid", fgColor="1F3864")
    header_font = Font(bold=True, color="FFFFFF")
    for sheet_name, df in [("Synthese", df_synth), ("Detail_boites", df_boxes_detail), ("Analyse_erreurs", df_err)]:
        ws = writer.sheets[sheet_name]
        for col_idx, col in enumerate(df.columns, start=1):
            cell = ws.cell(row=1, column=col_idx)
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(horizontal="center", vertical="center")
            # largeur : longue pour les colonnes de texte
            width = 60 if col in ("rapport_maira", "rapport_brut", "boites_predites", "boites_reelles_RSNA", "phrase") else 18
            from openpyxl.utils import get_column_letter
            ws.column_dimensions[get_column_letter(col_idx)].width = width
        ws.freeze_panes = "A2"  # fige la ligne d'en-tete

print(f"Fichier Excel cree : {out_path}")
print(f"  Feuille 'Synthese'      : {len(df_synth)} lignes (1 par image)")
print(f"  Feuille 'Detail_boites' : {len(df_boxes_detail)} lignes (1 par boite)")
print(f"  Feuille 'Analyse_erreurs': {len(df_err)} lignes (cas mal classes)")
print("\nApercu synthese :")
print(df_synth[["patientId","verite_terrain","classe_predite","nb_boites_predites"]].head(10).to_string(index=False))

## 12. Analyse des termes déclencheurs (diagnostic de la sur-détection)

Le principal défaut observé est que des cas `uncertain` sont classés `suspected_opacity`. Cette cellule identifie **quels termes d'opacité** déclenchent le plus ces erreurs, afin de savoir si un terme est trop large (candidat à retrait) ou si le problème est ailleurs (MAIRA-2 décrit réellement des findings sur ces images ambiguës).

In [ ]:
from collections import Counter

# Cas uncertain classes a tort en suspected_opacity
mis = [rec for rec in viz_records
       if rec["true"] == "uncertain" and rec["pred"] == "suspected_opacity"]

print(f"Cas 'uncertain' -> 'suspected_opacity' : {len(mis)}\n")

trigger_counter = Counter()
for rec in mis:
    report = rec["report"]
    for term in OPACITY_TERMS:
        if term in report and not _negated(report, term):
            trigger_counter[term] += 1

print("Termes d'opacite ayant declenche le faux positif (frequence) :")
for term, cnt in trigger_counter.most_common():
    print(f"  {term:<25} : {cnt}")

print("\nExemples de rapports mal classes (5 premiers) :")
for rec in mis[:5]:
    print(f"\n[{rec['pid']}]")
    print("  ", rec["report"][:260])

# Part des erreurs venant d'une BOITE sur pathologie vs du TEXTE seul
by_box = sum(1 for rec in mis if rec["box_on_patho"])
print(f"\nParmi ces {len(mis)} erreurs : {by_box} declenchees par une BOITE sur phrase pathologique, "
      f"{len(mis)-by_box} par le TEXTE seul.")
print("=> Si beaucoup sont dues aux boites, durcir le filtre de boites ; "
      "si dues au texte, MAIRA-2 decrit de vrais findings sur ces cas ambigus (limite intrinseque).")

## 13. Classification secondaire par LLM (GPT-OSS-120B via OpenRouter)

La classification par règles atteint ses limites sur les cas ambigus. On délègue ici l'interprétation du rapport MAIRA-2 à un LLM de raisonnement médical, **GPT-OSS-120B** (fort sur HealthBench), via OpenRouter.

**Principe.** Pour chaque image, on envoie au LLM le rapport ancré de MAIRA-2 (phrases + boîtes) et on lui demande de classer en `normal` / `suspected_opacity` / `uncertain`, en gérant les négations et les dispositifs médicaux. On compare ensuite à la vérité terrain, comme pour la classification par règles.

> Nécessite un secret Kaggle `OPENROUTER_API_KEY`. On réutilise `viz_records` (déjà calculé) : **pas besoin de relancer MAIRA-2**, on ne fait qu'analyser ses sorties.

In [ ]:
# OpenRouter s'utilise via le SDK OpenAI standard (base_url personnalisee).
!pip install -q openai 2>/dev/null

from openai import OpenAI
from kaggle_secrets import UserSecretsClient

OPENROUTER_KEY = UserSecretsClient().get_secret("OPENROUTER_API_KEY")
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_KEY,
)

LLM_MODEL = "openai/gpt-oss-120b:free"   # choix : fort raisonnement medical (HealthBench)

SYSTEM_PROMPT = """You are a radiology expert assisting an educational triage tool.
You receive the grounded findings of a frontal chest X-ray, as a list of
(finding_description, bounding_boxes) produced by an upstream model.

Classify the image into EXACTLY ONE of three classes:
- "suspected_opacity": the findings describe a pulmonary opacity, consolidation,
  infiltrate, pneumonia, effusion or similar airspace abnormality that is ASSERTED
  (not negated).
- "normal": the findings explicitly indicate clear lungs / no acute abnormality,
  OR describe only medical devices (PICC line, ET tube, NG tube, catheter) without
  any pulmonary abnormality.
- "uncertain": the findings are ambiguous, non-specific, or you cannot confidently
  decide between normal and opacity.

Important rules:
- Treat negated findings ("no focal infiltrate", "no effusion") as NORMAL evidence,
  not as opacity.
- A bounding box on a device (line, tube, catheter) is NOT an opacity.
- Prioritise patient safety: if a genuine airspace abnormality is asserted, prefer
  "suspected_opacity" over "uncertain".

Return ONLY a valid JSON object, no prose:
{"predicted_class": "...", "confidence": 0.0-1.0, "reason": "short justification"}"""

def parsed_to_prompt_text(parsed):
    """Formate la sortie MAIRA-2 (liste (phrase, boites)) en texte pour le LLM."""
    lines = []
    for item in parsed if isinstance(parsed, (list, tuple)) else []:
        if isinstance(item, (list, tuple)) and len(item) >= 1:
            phrase = str(item[0])
            boxes = item[1] if len(item) >= 2 else None
            lines.append(f"- {phrase}  boxes={boxes}")
    return "\n".join(lines) if lines else "(no findings)"

print("Client OpenRouter pret. Modele :", LLM_MODEL)

In [ ]:
import json as _json
import time

def classify_with_llm(parsed, max_retries=5):
    """Envoie le rapport MAIRA-2 au LLM. Retourne (classe, confiance, raison, success).
    - success=True  : le LLM a bien repondu (prediction fiable)
    - success=False : echec (429 apres retries, ou autre erreur) -> A EXCLURE des metriques
    Gestion du rate limit : retry avec back-off exponentiel (3, 6, 12, 24, 48 s)."""
    user_text = parsed_to_prompt_text(parsed)
    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model=LLM_MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_text},
                ],
                temperature=0,
                max_tokens=300,
            )
            content = resp.choices[0].message.content.strip()
            content = content.replace("```json", "").replace("```", "").strip()
            s, e = content.find("{"), content.rfind("}") + 1
            obj = _json.loads(content[s:e])
            cls = obj.get("predicted_class", "uncertain")
            if cls not in {"normal", "suspected_opacity", "uncertain"}:
                cls = "uncertain"
            return cls, obj.get("confidence", 0.0), obj.get("reason", ""), True
        except Exception as ex:
            msg = str(ex)
            if "429" in msg or "rate" in msg.lower():
                wait = (2 ** attempt) * 3   # 3, 6, 12, 24, 48 s
                time.sleep(wait)
                continue
            # erreur non liee au rate limit : inutile d'insister
            return "uncertain", 0.0, f"error: {ex}", False
    # rate limit persistant apres tous les retries
    return "uncertain", 0.0, "error: rate limit after retries", False

# --- Test sur un rapport avant de lancer les 100 ---
demo_rec = viz_records[0]
cls_llm, conf_llm, reason_llm, ok_llm = classify_with_llm(
    [(phrase, box) for phrase, box in demo_rec["pred_boxes"]]
)
print("Image      :", demo_rec["pid"])
print("Verite     :", demo_rec["true"])
print("Regles     :", demo_rec["pred"])
print("LLM        :", cls_llm, f"(conf={conf_llm}, success={ok_llm})")
print("Raison LLM :", reason_llm)
if not ok_llm:
    print("\\n[!] Le test a echoue (rate limit probable). Si ca persiste sur la boucle,")
    print("    ajoute une cle OpenRouter payante (quelques $) et relance.")

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from tqdm.auto import tqdm
import time

llm_preds, llm_trues, llm_reasons, llm_ok = [], [], [], []

for rec in tqdm(viz_records, total=len(viz_records)):
    parsed_like = [(phrase, box) for phrase, box in rec["pred_boxes"]]
    cls, conf, reason, ok = classify_with_llm(
        parsed_like if parsed_like else [(rec["report"], None)]
    )
    llm_preds.append(cls); llm_trues.append(rec["true"])
    llm_reasons.append(reason); llm_ok.append(ok)
    time.sleep(2.0)   # tier gratuit : espacement prudent (Levier 2 partiel)

# --- Compteur d'echecs : ESSENTIEL pour juger la fiabilite ---
mask = np.array(llm_ok)
n_ok, n_fail = int(mask.sum()), int((~mask).sum())
print(f"Appels reussis : {n_ok} / {len(mask)}   |   echecs (exclus) : {n_fail}")
if n_fail > 5:
    print("[!] Beaucoup d'echecs : metriques calculees sur un sous-ensemble reduit.")
    print("    Pour un run fiable, ajoute une cle OpenRouter payante et relance.\\n")

# --- Metriques UNIQUEMENT sur les appels reussis (les echecs ne sont PAS des predictions) ---
valid_preds = [p for p, o in zip(llm_preds, llm_ok) if o]
valid_trues = [t for t, o in zip(llm_trues, llm_ok) if o]

labels = ["normal", "suspected_opacity", "uncertain"]
if len(valid_preds) > 0:
    acc = accuracy_score(valid_trues, valid_preds)
    f1  = f1_score(valid_trues, valid_preds, labels=labels, average="macro")
    print("=== CLASSIFICATION SECONDAIRE (GPT-OSS-120B) — sur appels reussis uniquement ===")
    print(f"Base : {len(valid_preds)} images   Accuracy : {acc:.4f}   Macro-F1 : {f1:.4f}")
    print(classification_report(valid_trues, valid_preds, labels=labels, zero_division=0))
    cmn = confusion_matrix(valid_trues, valid_preds, labels=labels)
    print(pd.DataFrame(cmn, index=labels, columns=labels))
    op_i = labels.index("suspected_opacity")
    if cmn[op_i].sum():
        print(f"\\n>>> Rappel suspected_opacity (LLM) : {cmn[op_i,op_i]/cmn[op_i].sum():.4f}")

    # Comparaison regles vs LLM sur les MEMES images reussies
    rules_valid = [rec["pred"] for rec, o in zip(viz_records, llm_ok) if o]
    agree = np.mean([a == b for a, b in zip(rules_valid, valid_preds)])
    print(f"\\nAccord regles vs LLM (sur {len(valid_preds)} images) : {agree:.2%}")
else:
    print("Aucun appel reussi : impossible de calculer les metriques. Ajoute une cle payante.")

# --- Sauvegarde complete (avec flag succes pour tracabilite) ---
pd.DataFrame({
    "patientId": [r["pid"] for r in viz_records],
    "verite": llm_trues,
    "pred_regles": [r["pred"] for r in viz_records],
    "pred_LLM": llm_preds,
    "appel_reussi": llm_ok,
    "raison_LLM": llm_reasons,
    "rapport": [r["report"] for r in viz_records],
}).to_csv("/kaggle/working/maira2_llm_comparison.csv", index=False)
print("\\nSauvegarde : /kaggle/working/maira2_llm_comparison.csv")

## 14. Export final pour l'interface web (CSV complet avec boîtes)

Cette cellule produit `maira2_interface.csv`, pensé pour l'application web de secours.
Il contient, pour chaque image de la banque :
- la **classe finale** (celle du LLM GPT-OSS si l'appel a réussi, sinon celle des règles) ;
- les deux classes séparées (règles et LLM) pour référence ;
- les **boîtes prédites** par MAIRA-2, en pixels sur une image 1024×1024 (format standard RSNA) ;
- les **boîtes réelles** RSNA (vérité terrain), au même format ;
- le rapport MAIRA-2 et la vérité terrain.

Les boîtes sont stockées en JSON dans les cellules, pour être relues facilement par l'interface.

In [ ]:
import json as _json

# Dimension de reference pour l'interface (RSNA = 1024x1024)
REF = 1024

def pred_boxes_to_pixels(pred_boxes):
    """(phrase, boite normalisee 0-1) -> liste de dict pixels {x1,y1,x2,y2,phrase}."""
    out = []
    for phrase, (x1, y1, x2, y2) in pred_boxes:
        out.append({
            "x1": round(x1 * REF), "y1": round(y1 * REF),
            "x2": round(x2 * REF), "y2": round(y2 * REF),
            "phrase": phrase,
        })
    return out

def gt_rows_to_pixels(gt_rows):
    """Boites RSNA (x,y,w,h en pixels 1024) -> liste de dict {x1,y1,x2,y2}."""
    out = []
    for _, r in gt_rows.iterrows():
        out.append({
            "x1": int(r["x"]), "y1": int(r["y"]),
            "x2": int(r["x"] + r["width"]), "y2": int(r["y"] + r["height"]),
        })
    return out

# Construire le mapping patientId -> classe LLM (si dispo et appel reussi)
llm_by_pid = {}
try:
    for pid, pred, ok in zip([r["pid"] for r in viz_records], llm_preds, llm_ok):
        llm_by_pid[pid] = pred if ok else None
except NameError:
    # Si la section 13 (LLM) n'a pas ete executee, on se rabat sur les regles seules
    print("[i] Section LLM non executee : classe finale = regles uniquement.")

rows = []
for rec in viz_records:
    pid = rec["pid"]
    cls_rules = rec["pred"]
    cls_llm = llm_by_pid.get(pid)  # None si echec ou section LLM non lancee
    # Classe finale : priorite au LLM s'il a repondu, sinon regles
    cls_final = cls_llm if cls_llm is not None else cls_rules

    rows.append({
        "patientId": pid,
        "classe_finale": cls_final,
        "classe_regles": cls_rules,
        "classe_LLM": cls_llm if cls_llm is not None else "",
        "verite_terrain": rec["true"],
        "boites_predites": _json.dumps(pred_boxes_to_pixels(rec["pred_boxes"])),
        "boites_reelles": _json.dumps(gt_rows_to_pixels(rec["gt_rows"])),
        "rapport_maira": rec["report"],
        "ref_size": REF,
    })

df_interface = pd.DataFrame(rows)
out_csv = "/kaggle/working/maira2_interface.csv"
df_interface.to_csv(out_csv, index=False)
print(f"Export interface : {out_csv}  ({len(df_interface)} images)")
print("\nColonnes :", list(df_interface.columns))
print("\nApercu (2 premieres lignes) :")
for _, r in df_interface.head(2).iterrows():
    print(f"\n{r['patientId']} | finale={r['classe_finale']} | verite={r['verite_terrain']}")
    print("  boites predites :", r["boites_predites"][:150])
    print("  boites reelles  :", r["boites_reelles"][:150])